In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from itertools import combinations
import warnings, gc
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED    = 99          # Different seed from other notebooks for diversity!
N_FOLDS = 10
TARGET  = 'Irrigation_Need'
target_map  = {'Low': 0, 'Medium': 1, 'High': 2}
reverse_map = {0: 'Low', 1: 'Medium', 2: 'High'}

In [2]:
COMP = 'data/raw/'
ORIG = 'data/OG_data/'

train = pd.read_csv('/home/chotu/Downloads/Irrigation_Submission/data/raw/train.csv')
test  = pd.read_csv('/home/chotu/Downloads/Irrigation_Submission/data/raw/test.csv')
orig  = pd.read_csv('/home/chotu/Downloads/Irrigation_Submission/data/OG_data/irrigation_prediction.csv')

train[TARGET] = train[TARGET].map(target_map)
orig[TARGET]  = orig[TARGET].map(target_map)

CATS = ['Soil_Type','Crop_Type','Crop_Growth_Stage','Season',
        'Irrigation_Type','Water_Source','Mulching_Used','Region']
NUMS = ['Soil_pH','Soil_Moisture','Organic_Carbon','Electrical_Conductivity',
        'Temperature_C','Humidity','Rainfall_mm','Sunlight_Hours',
        'Wind_Speed_kmh','Field_Area_hectare','Previous_Irrigation_mm']
print(f'Train: {train.shape}, Test: {test.shape}, Orig: {orig.shape}')

Train: (630000, 21), Test: (270000, 20), Orig: (10000, 20)


In [3]:
def aggressive_fe(df_list):
    for df in df_list:
        eps = 1e-6
        # Core interactions
        df['water_avail']   = df['Soil_Moisture'] + df['Rainfall_mm'] / 300.0
        df['heat_stress']   = df['Temperature_C'] / (df['Soil_Moisture'] + eps)
        df['rain_cool']     = df['Rainfall_mm'] / (df['Temperature_C'] + eps)
        df['wind_evap']     = df['Wind_Speed_kmh'] * df['Temperature_C'] / 100.0
        df['soil_x_rain']   = df['Soil_Moisture'] * df['Rainfall_mm'] / 1000.0
        df['temp_x_dry']    = df['Temperature_C'] * (100 - df['Soil_Moisture']) / 100.0
        df['ec_x_ph']       = df['Electrical_Conductivity'] * df['Soil_pH']
        df['carbon_x_ph']   = df['Organic_Carbon'] * df['Soil_pH']
        df['sun_x_temp']    = df['Sunlight_Hours'] * df['Temperature_C']
        df['area_x_rain']   = df['Field_Area_hectare'] * df['Rainfall_mm']
        df['prev_ratio']    = df['Previous_Irrigation_mm'] / (df['Rainfall_mm'] + eps)
        # Threshold flags
        df['soil_lt_25']    = (df['Soil_Moisture'] < 25).astype(int)
        df['soil_lt_30']    = (df['Soil_Moisture'] < 30).astype(int)
        df['temp_gt_30']    = (df['Temperature_C'] > 30).astype(int)
        df['temp_gt_35']    = (df['Temperature_C'] > 35).astype(int)
        df['rain_lt_300']   = (df['Rainfall_mm'] < 300).astype(int)
        df['rain_lt_200']   = (df['Rainfall_mm'] < 200).astype(int)
        df['wind_gt_10']    = (df['Wind_Speed_kmh'] > 10).astype(int)
        df['wind_gt_15']    = (df['Wind_Speed_kmh'] > 15).astype(int)
        # Logit priors
        df['logit_low']     = (16.3173 - 11.0237*df['soil_lt_25'] - 5.8559*df['temp_gt_30']
                               - 10.8500*df['rain_lt_300'] - 5.8284*df['wind_gt_10'])
        df['logit_med']     = (4.6524  + 0.3290*df['soil_lt_25'] - 0.0204*df['temp_gt_30']
                               + 0.1542*df['rain_lt_300'] + 0.0841*df['wind_gt_10'])
        df['logit_high']    = (-20.9697 + 10.6947*df['soil_lt_25'] + 5.8763*df['temp_gt_30']
                               + 10.6958*df['rain_lt_300'] + 5.7444*df['wind_gt_10'])
        # Digit extraction on all numerics
        for c in NUMS:
            for k in range(-2, 3):
                df[f'{c}_d{k}'] = (df[c] // (10**k) % 10).astype(int)
    return df_list

aggressive_fe([train, test, orig])
print('Feature engineering done')

Feature engineering done


In [4]:
# 2-way and 3-way categorical combos
COMBOS = []
for c1, c2 in combinations(CATS, 2):
    col = f'{c1}__{c2}'
    for df in [train, test, orig]:
        df[col] = df[c1].astype(str) + '_' + df[c2].astype(str)
    COMBOS.append(col)

for c1, c2, c3 in combinations(CATS[:5], 3):  # limit 3-way to top 5 cats
    col = f'{c1}__{c2}__{c3}'
    for df in [train, test, orig]:
        df[col] = df[c1].astype(str) + '_' + df[c2].astype(str) + '_' + df[c3].astype(str)
    COMBOS.append(col)

print(f'Combo features: {len(COMBOS)}')

# Freq encoding
FREQ_FEATS = []
for c in CATS + COMBOS:
    freq = pd.concat([train[c], orig[c], test[c]]).value_counts(normalize=True)
    for df in [train, test, orig]:
        df[f'FREQ_{c}'] = df[c].map(freq).fillna(0).astype(float)
    FREQ_FEATS.append(f'FREQ_{c}')

# Label encode
for c in CATS + COMBOS:
    le = LabelEncoder()
    combined = pd.concat([train[c], test[c], orig[c]]).astype(str)
    le.fit(combined)
    for df in [train, test, orig]:
        df[c] = le.transform(df[c].astype(str)).astype('int32')

# Orig-based TE (leak-free) — applied to train, test AND orig
TE_FEATS = []
for c in CATS:
    for cls in [0, 1, 2]:
        col = f'TE_{c}_cls{cls}'
        te_map = orig.groupby(c)[TARGET].apply(lambda x: (x==cls).mean())
        for df in [train, test, orig]:   # <-- orig added
            df[col] = df[c].map(te_map).fillna(1/3)
        TE_FEATS.append(col)

# Orig stats TE — applied to train, test AND orig
ORIG_STATS = []
for c in CATS + NUMS[:5]:
    stats = orig.groupby(c)[TARGET].agg(['mean','std']).reset_index()
    stats.columns = [c, f'ORIG_{c}_mean', f'ORIG_{c}_std']
    train = train.merge(stats, on=c, how='left').fillna({f'ORIG_{c}_mean':1.0, f'ORIG_{c}_std':0.0})
    test  = test.merge(stats,  on=c, how='left').fillna({f'ORIG_{c}_mean':1.0, f'ORIG_{c}_std':0.0})
    orig  = orig.merge(stats,  on=c, how='left').fillna({f'ORIG_{c}_mean':1.0, f'ORIG_{c}_std':0.0})
    ORIG_STATS += [f'ORIG_{c}_mean', f'ORIG_{c}_std']

# Assemble features
DIG_FEATS  = [c for c in train.columns if '_d-' in c or '_d0' in c or '_d1' in c or '_d2' in c]
NEW_NUMS   = ['water_avail','heat_stress','rain_cool','wind_evap','soil_x_rain','temp_x_dry',
              'ec_x_ph','carbon_x_ph','sun_x_temp','area_x_rain','prev_ratio',
              'soil_lt_25','soil_lt_30','temp_gt_30','temp_gt_35','rain_lt_300',
              'rain_lt_200','wind_gt_10','wind_gt_15','logit_low','logit_med','logit_high']

ALL_FEATS = list(dict.fromkeys(
    NUMS + NEW_NUMS + CATS + COMBOS + FREQ_FEATS + TE_FEATS + DIG_FEATS
))
print(f'Total unique features: {len(ALL_FEATS)}')

Combo features: 38
Total unique features: 204


In [5]:
X     = train[ALL_FEATS].values.astype(np.float32)
y     = train[TARGET].values
X_te  = test[ALL_FEATS].values.astype(np.float32)
X_orig = orig[ALL_FEATS].values.astype(np.float32)
y_orig = orig[TARGET].values

# Quick 3-fold search on a subset for speed
skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

def optuna_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 500, 3000),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 0.5),
        'alpha':            trial.suggest_float('alpha', 0, 5),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5),
        'max_leaves':       trial.suggest_int('max_leaves', 15, 127),
        'objective': 'multi:softprob', 'num_class': 3,
        'eval_metric': 'mlogloss', 'device': 'cuda',
        'random_state': SEED, 'n_jobs': -1,
    }
    scores = []
    for tr_idx, val_idx in skf3.split(X, y):
        sw = compute_sample_weight('balanced', y[tr_idx])
        m = xgb.XGBClassifier(**params)
        m.fit(X[tr_idx], y[tr_idx], sample_weight=sw, verbose=False)
        p = m.predict_proba(X[val_idx])
        scores.append(balanced_accuracy_score(y[val_idx], p.argmax(1)))
    return np.mean(scores)

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
study.optimize(optuna_objective, n_trials=50, show_progress_bar=True)
best_params = study.best_params
best_params.update({'objective':'multi:softprob','num_class':3,'eval_metric':'mlogloss',
                    'device':'cuda','random_state':SEED,'n_jobs':-1})
print(f'Best Optuna OOF: {study.best_value:.5f}')
print('Best params:', best_params)

  0%|          | 0/50 [00:00<?, ?it/s]

Best Optuna OOF: 0.97171
Best params: {'n_estimators': 2608, 'max_depth': 3, 'learning_rate': 0.028485859665133045, 'subsample': 0.8690341916769421, 'colsample_bytree': 0.9221801171672752, 'min_child_weight': 1, 'gamma': 0.22764048509089158, 'alpha': 1.0483671800101981, 'reg_lambda': 1.6121896705932026, 'max_leaves': 110, 'objective': 'multi:softprob', 'num_class': 3, 'eval_metric': 'mlogloss', 'device': 'cuda', 'random_state': 99, 'n_jobs': -1}


In [6]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_probs  = np.zeros((len(X), 3))
test_probs = np.zeros((len(X_te), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'Fold {fold+1}/{N_FOLDS}', end=' ')
    Xtr = np.vstack([X[tr_idx], X_orig])
    ytr = np.concatenate([y[tr_idx], y_orig])
    sw  = compute_sample_weight('balanced', ytr)

    model = xgb.XGBClassifier(**best_params)
    model.fit(Xtr, ytr, sample_weight=sw, verbose=False)

    oof_probs[val_idx]  = model.predict_proba(X[val_idx])
    test_probs         += model.predict_proba(X_te) / N_FOLDS
    score = balanced_accuracy_score(y[val_idx], oof_probs[val_idx].argmax(1))
    print(f'| BA={score:.5f}')
    del model; gc.collect()

oof_score = balanced_accuracy_score(y, oof_probs.argmax(1))
print(f'\nOOF Balanced Accuracy: {oof_score:.5f}')

Fold 1/10 | BA=0.97143
Fold 2/10 | BA=0.97197
Fold 3/10 | BA=0.97123
Fold 4/10 | BA=0.97251
Fold 5/10 | BA=0.97409
Fold 6/10 | BA=0.97489
Fold 7/10 | BA=0.97216
Fold 8/10 | BA=0.97277
Fold 9/10 | BA=0.97297
Fold 10/10 | BA=0.97308

OOF Balanced Accuracy: 0.97271


In [7]:
def accuracy_by_class(y_true, probs):
    preds = np.argmax(probs, axis=1)
    return balanced_accuracy_score(y_true, preds)

def cw_objective(trial):
    cw = np.array([trial.suggest_float(f'cw{i}', 0.3, 3.0) for i in range(3)])
    adj = oof_probs * cw
    adj /= adj.sum(axis=1, keepdims=True)
    return accuracy_by_class(y, adj)

cw_study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
cw_study.optimize(cw_objective, n_trials=500, show_progress_bar=True)

best_cw = np.array([cw_study.best_params[f'cw{i}'] for i in range(3)])
print(f'Best class weights: Low={best_cw[0]:.4f}, Medium={best_cw[1]:.4f}, High={best_cw[2]:.4f}')

final_probs = test_probs * best_cw
final_probs /= final_probs.sum(axis=1, keepdims=True)

oof_adj = oof_probs * best_cw
oof_adj /= oof_adj.sum(axis=1, keepdims=True)
print(f'OOF after CW tuning: {balanced_accuracy_score(y, oof_adj.argmax(1)):.5f}')

  0%|          | 0/500 [00:00<?, ?it/s]

Best class weights: Low=1.1764, Medium=1.7134, High=2.7929
OOF after CW tuning: 0.97390


In [9]:
sub = pd.read_csv('/home/chotu/Downloads/Irrigation_Submission/data/raw/sample_submission.csv')
sub[TARGET] = [reverse_map[p] for p in final_probs.argmax(1)]
sub.to_csv('submission_xgb_optuna.csv', index=False)
print('submission_xgb_optuna.csv saved')
print(sub[TARGET].value_counts())

# Soft probabilities (USE THESE FOR ENSEMBLING!)
proba_df = pd.DataFrame(final_probs, columns=['proba_Low','proba_Medium','proba_High'])
proba_df.insert(0, 'id', sub['id'].values)
proba_df.to_csv('soft_proba_xgb_optuna.csv', index=False)
print('soft_proba_xgb_optuna.csv saved — use this for soft voting ensemble!')

oof_df = pd.DataFrame(oof_adj, columns=['oof_Low','oof_Medium','oof_High'])
oof_df['true'] = y
oof_df.to_csv('oof_xgb_optuna.csv', index=False)

submission_xgb_optuna.csv saved
Irrigation_Need
Low       159843
Medium     99481
High       10676
Name: count, dtype: int64
soft_proba_xgb_optuna.csv saved — use this for soft voting ensemble!
